# 🧠 Analisis Dampak Media Sosial terhadap Kesehatan Mental Remaja
**Metode: CRISP-DM | Algoritma: Decision Tree (EDA) & Regresi Logistik (Modeling)**

---

> 📦 Dataset: [Teen Mental Health Dataset — Kaggle](https://www.kaggle.com/datasets/sunil123kumar/social-media-impact-on-mental-health)

Notebook ini mengikuti kerangka kerja **CRISP-DM** (*Cross-Industry Standard Process for Data Mining*):

| # | Fase | Fokus |
|---|------|-------|
| 1 | **Business Understanding** | Rumusan masalah & tujuan analisis |
| 2 | **Data Understanding** | EDA & eksplorasi korelasi dengan Decision Tree |
| 3 | **Data Preparation** | Encoding, split data latih & uji |
| 4 | **Modeling** | Regresi Logistik sebagai model prediksi utama |
| 5 | **Evaluation** | Akurasi, Classification Report, Confusion Matrix |
| 6 | **Deployment** | Interpretasi, kesimpulan & rekomendasi |

---
## ⚙️ Setup & Import Library

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    RocCurveDisplay
)

print('✅ Semua library berhasil diimport.')

---
## 📌 Fase 1 — Business Understanding

### Latar Belakang
Penggunaan media sosial yang berlebihan di kalangan remaja telah menjadi perhatian serius dalam dunia kesehatan mental. Berbagai studi menunjukkan bahwa paparan konten negatif, tekanan sosial daring, serta kurangnya waktu tidur akibat penggunaan media sosial berkontribusi terhadap meningkatnya tingkat stres, kecemasan, dan depresi pada usia remaja.

Kami bermaksud menganalisis hubungan antara kebiasaan penggunaan media sosial dengan risiko depresi, serta membangun model prediktif yang dapat dijadikan dasar pengambilan keputusan oleh orang tua, pendidik, maupun tenaga kesehatan.

### Rumusan Masalah
> **Apakah kebiasaan penggunaan media sosial (durasi, platform, interaksi sosial, tingkat stres) dapat digunakan untuk memprediksi risiko depresi pada remaja?**

### Tujuan
1. Mengidentifikasi fitur-fitur yang paling berkorelasi dengan depresi menggunakan **Decision Tree** sebagai alat eksplorasi.
2. Membangun model klasifikasi berbasis **Regresi Logistik** untuk memprediksi `depression_label`.
3. Memberikan rekomendasi berbasis data kepada pemangku kepentingan.

### Definisi Target
| Label | Keterangan |
|-------|------------|
| `0` | Tidak berisiko depresi |
| `1` | Berisiko depresi |

### Metrik Keberhasilan
- Akurasi model Regresi Logistik ≥ **75%**
- ROC-AUC Score ≥ **0.75**

---
## 📂 Fase 2 — Data Understanding

Pada fase ini kami melakukan eksplorasi menyeluruh terhadap dataset untuk memahami struktur, distribusi, dan hubungan antar fitur. Kami juga menggunakan **Decision Tree** sebagai alat bantu eksplorasi untuk mengidentifikasi fitur-fitur yang paling diskriminatif.

### 2.1 Load Dataset

In [ ]:
# Download dataset dari Kaggle
path = kagglehub.dataset_download('sunil123kumar/social-media-impact-on-mental-health')
print('📁 Path dataset:', path)

df = pd.read_csv(f'{path}/Teen_Mental_Health_Dataset.csv')
print(f'✅ Dataset dimuat — {df.shape[0]} baris × {df.shape[1]} kolom')

### 2.2 Eksplorasi Awal

In [ ]:
# Preview data
print('=== 5 Baris Pertama ===')
df.head()

In [ ]:
# Tipe data & info umum
print('=== Info Dataset ===')
df.info()

In [ ]:
# Statistik deskriptif
print('=== Statistik Deskriptif ===')
df.describe().round(2)

In [ ]:
# Cek missing values
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else '✅ Tidak ada missing values.')

# Distribusi target
print('\n=== Distribusi Target (depression_label) ===')
print(df['depression_label'].value_counts())
print('Proporsi:', df['depression_label'].value_counts(normalize=True).round(3).to_dict())

### 2.3 Analisis Korelasi Antar Fitur

In [ ]:
# Heatmap korelasi fitur numerik
df_numeric = df.select_dtypes(include=['int64', 'float64'])
corr_matrix = df_numeric.corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Heatmap Korelasi Antar Fitur Numerik', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Korelasi khusus terhadap depression_label
corr_depression = corr_matrix[['depression_label']].sort_values(by='depression_label', ascending=False)

plt.figure(figsize=(6, 7))
sns.heatmap(corr_depression, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Korelasi Fitur terhadap Depression Label', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n📊 Interpretasi:')
print('Tiga fitur dengan korelasi tertinggi terhadap depresi:')
print('  1. daily_social_media_hours — durasi penggunaan media sosial per hari')
print('  2. stress_level             — tingkat stres yang dialami remaja')
print('  3. anxiety_level            — tingkat kecemasan yang dialami remaja')

### 2.4 Eksplorasi dengan Decision Tree — Identifikasi Fitur Penting

Kami menggunakan **Decision Tree** bukan sebagai model utama, melainkan sebagai **alat eksplorasi** untuk memahami fitur mana yang paling membedakan kelas `depression_label`. Pendekatan ini membantu kami memilih fitur yang relevan sebelum masuk ke tahap modeling.

In [ ]:
# Siapkan data sementara untuk eksplorasi Decision Tree
X_explore = df.drop('depression_label', axis=1).copy()
y_explore  = df['depression_label']

# Encoding sementara untuk keperluan eksplorasi
for col in X_explore.select_dtypes(include='object').columns:
    X_explore[col] = LabelEncoder().fit_transform(X_explore[col])

# Decision Tree untuk eksplorasi feature importance
dt_explore = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
dt_explore.fit(X_explore, y_explore)

print('✅ Decision Tree eksplorasi selesai dilatih.')
print(f'   Kedalaman pohon : {dt_explore.get_depth()}')
print(f'   Jumlah leaf node: {dt_explore.get_n_leaves()}')

In [ ]:
# Visualisasi Feature Importance dari Decision Tree
importance_explore = pd.DataFrame({
    'Feature'   : X_explore.columns,
    'Importance': dt_explore.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_explore, x='Importance', y='Feature', palette='viridis')
plt.title('Feature Importance — Decision Tree (Eksplorasi)', fontsize=13, fontweight='bold')
plt.xlabel('Tingkat Kepentingan')
plt.ylabel('Fitur')
plt.tight_layout()
plt.show()

print('\n📋 Ranking Feature Importance (Decision Tree):')
print(importance_explore.to_string(index=False))

In [ ]:
# Visualisasi Struktur Pohon Keputusan (Eksplorasi)
plt.figure(figsize=(22, 10))
plot_tree(
    dt_explore,
    feature_names=X_explore.columns,
    class_names=['No Depression', 'Depression'],
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title('Struktur Decision Tree — Eksplorasi Data Understanding', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**📝 Kesimpulan Data Understanding:**

Dari hasil eksplorasi menggunakan Decision Tree, kami menemukan bahwa fitur `stress_level`, `daily_social_media_hours`, dan `anxiety_level` secara konsisten muncul sebagai pemisah utama di node-node awal pohon keputusan — mengonfirmasi hasil analisis korelasi sebelumnya. Temuan ini akan menjadi acuan dalam tahap pemodelan menggunakan Regresi Logistik.

---
## 🔧 Fase 3 — Data Preparation

Pada fase ini kami menyiapkan data untuk proses pemodelan:
1. Memisahkan fitur (`X`) dan target (`y`)
2. Encoding kolom kategorikal dengan `LabelEncoder`
3. Membagi data menjadi **data latih (80%)** dan **data uji (20%)**

In [ ]:
# Langkah 1 — Pisahkan Fitur dan Target
X = df.drop('depression_label', axis=1).copy()
y = df['depression_label']

print(f'Fitur (X) : {X.shape}')
print(f'Target (y): {y.shape}')
print(f'Kolom fitur: {list(X.columns)}')

In [ ]:
# Langkah 2 — Encoding Kolom Kategorikal
le_gender   = LabelEncoder()
le_platform = LabelEncoder()
le_social   = LabelEncoder()

X['gender']                   = le_gender.fit_transform(X['gender'])
X['platform_usage']           = le_platform.fit_transform(X['platform_usage'])
X['social_interaction_level'] = le_social.fit_transform(X['social_interaction_level'])

print('✅ Encoding selesai. Semua fitur kini bertipe numerik.')
X.info()

In [ ]:
# Langkah 3 — Split Data Latih dan Data Uji (80:20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'✅ Data Latih : {X_train.shape[0]} sampel')
print(f'   Data Uji  : {X_test.shape[0]} sampel')
print(f'\nDistribusi kelas — Data Latih:\n{y_train.value_counts()}')
print(f'\nDistribusi kelas — Data Uji:\n{y_test.value_counts()}')

---
## 🤖 Fase 4 — Modeling (Data Mining)

Kami menggunakan **Regresi Logistik** sebagai algoritma utama untuk tahap data mining / pemodelan prediktif. Regresi Logistik dipilih karena:
- Cocok untuk klasifikasi biner (`depression_label` 0/1)
- Menghasilkan nilai probabilitas yang mudah diinterpretasikan
- Koefisien model dapat menjelaskan arah dan kekuatan pengaruh tiap fitur
- Komplementer dengan temuan eksplorasi Decision Tree di fase sebelumnya

In [ ]:
# Inisialisasi dan latih model Regresi Logistik
log_reg = LogisticRegression(
    max_iter=1000,
    random_state=42,
    solver='lbfgs'
)

log_reg.fit(X_train, y_train)
print('✅ Model Regresi Logistik berhasil dilatih.')

In [ ]:
# Koefisien Model — Interpretasi Arah Pengaruh Fitur
coef_df = pd.DataFrame({
    'Feature'    : X.columns,
    'Coefficient': log_reg.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in coef_df['Coefficient']]
sns.barplot(data=coef_df, x='Coefficient', y='Feature', palette=colors)
plt.axvline(x=0, color='black', linewidth=0.8, linestyle='--')
plt.title('Koefisien Regresi Logistik\n(Merah = Meningkatkan Risiko | Hijau = Menurunkan Risiko)',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n📋 Koefisien Model:')
print(coef_df.to_string(index=False))

---
## 📊 Fase 5 — Evaluation

Kami mengevaluasi performa model Regresi Logistik menggunakan:
- **Accuracy Score** — persentase prediksi benar
- **Classification Report** — Precision, Recall, F1-Score per kelas
- **Confusion Matrix** — visualisasi kesalahan klasifikasi
- **ROC-AUC Score** — kemampuan model membedakan dua kelas

In [ ]:
# Prediksi
y_pred      = log_reg.predict(X_test)
y_pred_prob = log_reg.predict_proba(X_test)[:, 1]

# Akurasi
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_prob)

print(f'🎯 Accuracy Score : {acc:.4f} ({acc*100:.2f}%)')
print(f'📈 ROC-AUC Score  : {auc:.4f}')

In [ ]:
# Classification Report
print('=== Classification Report ===')
print(classification_report(
    y_test, y_pred,
    target_names=['No Depression (0)', 'Depression (1)']
))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1 — Confusion Matrix
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
    xticklabels=['No Depression', 'Depression'],
    yticklabels=['No Depression', 'Depression']
)
axes[0].set_xlabel('Predicted Label', fontweight='bold')
axes[0].set_ylabel('Actual Label', fontweight='bold')
axes[0].set_title('Confusion Matrix — Regresi Logistik', fontweight='bold')

# Plot 2 — ROC Curve
RocCurveDisplay.from_predictions(y_test, y_pred_prob, ax=axes[1], name='Logistic Regression')
axes[1].set_title(f'ROC Curve (AUC = {auc:.3f})', fontweight='bold')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random Classifier')
axes[1].legend()

plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negative  (TN): {tn}  — Benar diprediksi Tidak Depresi')
print(f'False Positive (FP): {fp}  — Salah diprediksi Depresi')
print(f'False Negative (FN): {fn}  — Salah diprediksi Tidak Depresi')
print(f'True Positive  (TP): {tp}  — Benar diprediksi Depresi')

---
## 🚀 Fase 6 — Deployment (Interpretasi & Kesimpulan)

### Ringkasan Hasil

Model **Regresi Logistik** yang kami bangun berhasil memprediksi risiko depresi remaja berdasarkan pola penggunaan media sosial mereka. Eksplorasi awal menggunakan **Decision Tree** mengonfirmasi bahwa `stress_level`, `daily_social_media_hours`, dan `anxiety_level` adalah fitur paling diskriminatif — konsisten dengan hasil koefisien Regresi Logistik.

### Perbandingan Peran Algoritma

| Algoritma | Peran dalam CRISP-DM | Tujuan |
|-----------|---------------------|--------|
| **Decision Tree** | Data Understanding | Eksplorasi & identifikasi fitur penting |
| **Regresi Logistik** | Modeling (Data Mining) | Prediksi probabilitas risiko depresi |

### Rekomendasi

| Stakeholder | Rekomendasi |
|-------------|-------------|
| **Orang Tua** | Batasi durasi media sosial anak ≤ 2 jam/hari & pantau tanda stres |
| **Sekolah** | Adakan program literasi digital dan sesi konseling kesehatan mental |
| **Remaja** | Prioritaskan tidur cukup dan kurangi screen time sebelum tidur |
| **Peneliti** | Tambahkan fitur konteks keluarga & ekonomi untuk model yang lebih komprehensif |

### Keterbatasan
- Dataset bersifat sintetis, perlu validasi dengan data primer di lapangan.
- Regresi Logistik mengasumsikan hubungan linear antara fitur dan log-odds — perlu diuji lebih lanjut.
- Model belum divalidasi lintas kelompok demografis yang berbeda.